![alt text](./Cerny_logo_1.jpg)

# Analysis of Cerny ventilation recordings

## Further processing and analysis of ventilator parameters 

This notebook import the preprocessed ventilator data from piclkle archive and analyses all the ventilator parameter data and alarms data obtained with **0.5Hz sampling rate** from the Fabian ventilators at the Cerny neonatal transport service. It exports desrciptive statistics into Excel files and the further processed data as pickle archive.

- Total: **914 cases** with ventilator data available (with >=5 minutes recording time)
- Clinical data were not available for **95 cases** 
- Appropriate ventilator and clinical data are available for **819 cases**

The data processed and analysed in this Notebook were collected by the **Neonatal Emergency and Transport Service of the Peter Cerny Foundation**, Budapest, Hungary

**Author: Dr Gusztav Belteki**

### 1. Import the required libraries and set options

In [1]:
import IPython
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

import os
import sys
import pickle

from pandas import Series, DataFrame
from datetime import datetime, timedelta
from matplotlib import dates

%matplotlib inline
matplotlib.style.use('classic')
matplotlib.rcParams['figure.facecolor'] = 'w'

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)
# This is to turn off a warning message which is given when read_Excel() imports '.xlsx' files
import warnings
warnings.simplefilter("ignore")

In [2]:
print("Python version: {}".format(sys.version))
print("pandas version: {}".format(pd.__version__))
print("matplotlib version: {}".format(matplotlib.__version__))
print("NumPy version: {}".format(np.__version__))
print("IPython version: {}".format(IPython.__version__))

Python version: 3.12.11 | packaged by conda-forge | (main, Jun  4 2025, 14:38:53) [Clang 18.1.8 ]
pandas version: 2.3.2
matplotlib version: 3.10.6
NumPy version: 1.26.4
IPython version: 9.5.0


### 2. List and set the working directory and the directory to write out data

In [62]:
# Name of the external hard drive
DRIVE = 'GB_1'

# Path to project folder containing ventilation research results
PATH = os.path.join(os.sep, 'Users', 'guszti', 'Library', 'Mobile Documents', 'com~apple~CloudDocs', 
                            'Documents', 'Research', 'Ventilation')

# Folder to export the result of analysis
DIR_WRITE = os.path.join(PATH, 'ventilation_fabian', 'Analyses')
os.makedirs(DIR_WRITE, exist_ok = True)

# Folder on external drive to export data to and to import processed data exported by other Notebooks
DATA_DUMP = os.path.join(os.sep, 'Volumes', DRIVE, 'data_dump', 'fabian',)
os.makedirs(DATA_DUMP, exist_ok = True)

DIR_READ, DIR_WRITE, DATA_DUMP

('/Volumes/GB_1/Fabian/fabian_patient_data_all',
 '/Users/guszti/Library/Mobile Documents/com~apple~CloudDocs/Documents/Research/Ventilation/ventilation_fabian/Analyses',
 '/Volumes/GB_1/data_dump/fabian')

### 3. Import pickle archives

In [11]:
%%time

with open(os.path.join(DATA_DUMP, 'data_pars_1_150.pickle'), 'rb') as handle:
    data_pars_1_150 = pickle.load(handle)
with open(os.path.join(DATA_DUMP, 'data_pars_151_300.pickle'), 'rb') as handle:
    data_pars_151_300 = pickle.load(handle)
with open(os.path.join(DATA_DUMP, 'data_pars_301_450.pickle'), 'rb') as handle:
    data_pars_301_450 = pickle.load(handle)  
with open(os.path.join(DATA_DUMP, 'data_pars_451_600.pickle'), 'rb') as handle:
    data_pars_451_600 = pickle.load(handle)  
with open(os.path.join(DATA_DUMP, 'data_pars_601_750.pickle'), 'rb') as handle:
    data_pars_601_750 = pickle.load(handle)  
with open(os.path.join(DATA_DUMP, 'data_pars_751_900.pickle'), 'rb') as handle:
    data_pars_751_900 = pickle.load(handle)
with open(os.path.join(DATA_DUMP, 'data_pars_901_1050.pickle'), 'rb') as handle:
    data_pars_901_1050 = pickle.load(handle)  
with open(os.path.join(DATA_DUMP, 'data_pars_1051_1100.pickle'), 'rb') as handle:
    data_pars_1051_1100 = pickle.load(handle)
    
data_pars = {**data_pars_1_150, **data_pars_151_300, **data_pars_301_450, **data_pars_451_600,
             **data_pars_601_750, **data_pars_751_900, **data_pars_901_1050, **data_pars_1051_1100,}

CPU times: user 5.15 s, sys: 836 ms, total: 5.99 s
Wall time: 6.09 s


In [13]:
# Total number of ventilator recordings which are >=5 minutes
len(data_pars)

914

In [15]:
# Import clinical data
with open(os.path.join(DATA_DUMP, 'clin_df_1_1100.pickle'), 'rb') as handle:
    clin_df = pickle.load(handle)
len(clin_df)

819

### 4. Limit data to cases for which both clinical data and >=5 minutes of ventilator data are available

In [18]:
combined = sorted(set(list(clin_df.index)) & set(data_pars.keys()))
data_pars = {key : value for key, value in data_pars.items() if key in combined}
len(data_pars)

819

### 5. Create a dictionary of Dataframes with measured ventilator parameters only

In [21]:
ventilator_measurements = ['PIP', 'MAP', 'PEEP', 'Ti_PSV', 'Cdyn', 'C20_C', 'R', 'MV', 'MVresp', 
                           'VTemand', 'VTemand_resp', 'VTespon_pat', 'Leak', 'RR', 'Trigger', 'VTimand', 'FiO2',
                           'Flow_demand', 'Flow_insp', 'Flow_exp',]

In [23]:
data_pars_measurements = {}
for case in data_pars:
    data_pars_measurements[case] = data_pars[case][ventilator_measurements].copy()

In [25]:
# Replace textual data with np.nan
repl_dct = {'not valid': np.nan, 'out of range': np.nan, 'unused': np.nan}
for case in data_pars_measurements:
    data_pars_measurements[case].replace(repl_dct, inplace = True)

In [29]:
# Normalize relevant parameters to body weight
pars_per_kg = ['MV', 'VTimand', 'VTemand', 'VTespon_pat', 'VTemand_resp']
for case in data_pars_measurements:
    for par in pars_per_kg:
        data_pars_measurements[case]['%s_kg' % par] = \
        data_pars_measurements[case][par] / (clin_df.loc[case]['Weight'] / 1000)

In [31]:
# Drop columns which only have NaN values
for case in data_pars_measurements:
    data_pars_measurements[case].dropna(axis = 1, how = 'all', inplace = True)

##### Export dictionary containing measured ventilator parameters to a pickle archive

In [34]:
with open(os.path.join(DATA_DUMP, 'data_pars_measurements_1_1100.pickle'), 'wb') as handle:
    pickle.dump(data_pars_measurements, handle, protocol=pickle.HIGHEST_PROTOCOL)

### 6. Create a dictionary of Dataframes with ventilator settings

In [36]:
ventilator_settings = ['Patient_range', 'Ventilator_mode', 'PIP_set', 'PEEP_set', 'PIP_set_PSV', 
'FiO2_set', 'Flow_insp_set','Slope_set', 'Flow_exp_set', 'Ti_set', 'Te_set', 'RR_set', 
'IE_I_set', 'IE_E_set', 'Volume_limit_set', 'VG_set', 'Term_criteria_PSV_set', 'Apnea_time_set', 
'RR_backup_set', 'Trigger_sens_set', 'Powerstate', 'MV_lim_high_set',
'MV_lim_low_set', 'PIP_lim_high_set', 'PIP_lim_low_set', 'RR_lim_set', 'Leakage_lim_set',
'Measuring_unit_pressure_set', 'Flow_sensor_state', 'Oxy_sensor_state',
'P_man_breath_CPAP_HFO_set', 'P_man_breath_duoPAP_NCPAP_set', 'FiO2_flush_time_set', 'FiO2_flush_set',
'Ventilation_stopped', 'VG_state', 'Volume_limit_state', 'Ventilator_range', 'Trigger_mode',
'Pressure_rise_control']

In [38]:
data_pars_settings = {}
for case in data_pars:
    data_pars_settings[case] = data_pars[case][ventilator_settings].copy()

In [40]:
# Replace textual data with np.nan
repl_dct = {'not valid': np.nan, 'out of range': np.nan, 'unused': np.nan}
for case in data_pars_settings:
    data_pars_settings[case].replace(repl_dct, inplace = True)

In [42]:
# Normalize relevant parameters to body weight
pars_per_kg = ['Volume_limit_set', 'VG_set', 'MV_lim_high_set', 'MV_lim_low_set',]

for case in data_pars_settings:
    for par in pars_per_kg:
        dta = data_pars_settings[case][par]
        # Exclude data points when the parameter was 'off'
        data_pars_settings[case]['%s_kg' % par] = \
        dta[dta != 'off'] / (clin_df.loc[case]['Weight'] / 1000)

In [44]:
# Drop columns which only have NaN values
for case in data_pars_settings:
    data_pars_settings[case].dropna(axis=1, how='all', inplace = True)

##### Export dictionary containing ventilator settings to a pickle archive

In [47]:
with open(os.path.join(DATA_DUMP, 'data_pars_settings_1_1100.pickle'), 'wb') as handle:
    pickle.dump(data_pars_settings, handle, protocol=pickle.HIGHEST_PROTOCOL)

### 7. Create a dictionary of Dataframes with ventilator alarms

In [50]:
ventilator_alarms = ['Alarm_susp', 'Alarm_Flat_battery', 'Alarm_Checksum_ctrl_PIC', 'Alarm_Checksum_monitor_PIC',
'Alarm_Safety_relay_defect', 'Alarm_Sens_dev_prox_pressure', 'Alarm_input_pressure_blender', 'Alarm_excess_pressure',
'Alarm_voltage_monit', 'Alarm_SPI_interface', 'Alarm_DIO2_interface', 'Alarm_COM_interface', 'Alarm_I2C_interface',
'Alarm_parallel_interface', 'Alarm_serial_tem_interface', 'Alarm_low_physical_memory', 'Alarm_Fan_defect',
'Alarm_CO2_interface', 'Alarm_blender_defect', 'Alarm_battery_defect', 'Alarm_input_pressure_O2_supply',
'Alarm_input_pressure_air_supply', 'Alarm_tube_occlusion', 'Alarm_patient_disconnected', 'Alarm_ETT_blocked',
'Alarm_flow_sensor_defect', 'Alarm_flow_sensor_clean', 'Alarm_flow_sensor_disconnected', 'Oxygen_sensor_defect',
'Oxygen_sensor_used_up', 'Oxyen_value_divergence', 'Alarm_O2_sensor_cal_error', 'Alarm_MV_high', 'Alarm_MV_low',
'Alarm_pressure_high', 'Alarm_pressure_low', 'Alarm_PEEP_high', 'Alarm_RR_high', 'Alarm_ETT_leak_high','Alarm_apnea',
'Alarm_DCO2_high', 'Alarm_DCO2_low', 'Alarm_etCO2_high','Alarm_etCO2_low', 'Alarm_PIP_not_reached',
'Alarm_limited_volume', 'Alarm_volume_not_reached', 'Alarm_power_failure', 'Alarm_charge_battery_60min',
'Alarm_charge_battery_30min', 'Alarm_charge_battery_15min', 'Alarm_nebulizer_disconnection',
'Alarm_nebulizer_system_error', 'Alarm_CO2_module_not_connected', 'Alarm_CO2_filterline_not_connected',
'Alarm_CO2_check_sampleline', 'Alarm_CO2_check_airway_adapter', 'Alarm_CO2_sensor_faulty',]

In [52]:
data_pars_alarms = {}
for case in data_pars:
    data_pars_alarms[case] = data_pars[case][ventilator_alarms].copy()

In [54]:
# Replace textual data with np.nan
repl_dct = {'off': np.nan, 'not valid': np.nan, 'out of range': np.nan, 'unused': np.nan}
for case in data_pars_alarms:
    data_pars_alarms[case].replace(repl_dct, inplace = True)

In [56]:
# Drop columns which only have NaN values (these alarms never went off)
for case in data_pars_alarms:
    for column in data_pars_alarms[case].columns:
        if data_pars_alarms[case][column].sum() == 0:
            del data_pars_alarms[case][column]        

#### Export dictionary containing ventilator alarms to a pickle archive

In [58]:
with open(os.path.join(DATA_DUMP, 'data_pars_alarms_1_1100.pickle'), 'wb') as handle:
    pickle.dump(data_pars_alarms, handle, protocol=pickle.HIGHEST_PROTOCOL)